## 1. Imports

In [21]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
import re
import nltk
from nltk.corpus import stopwords
import os

nltk.data.path.append(r"C:\Users\hp\Desktop\DS1_Tweets.ID_886380-1\src\nltk_data")

import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [22]:
def load_tweets(path):
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    return [t.strip() for t in content.split(',') if t.strip()]

neg = load_tweets('data/processedNegative.csv')
neu = load_tweets('data/processedNeutral.csv')
pos = load_tweets('data/processedPositive.csv')

tweets = neg + neu + pos
labels = ['negative']*len(neg) + ['neutral']*len(neu) + ['positive']*len(pos)

df = pd.DataFrame({'tweet': tweets, 'label': labels})
df['label'].value_counts()

label
neutral     1566
positive    1183
negative    1116
Name: count, dtype: int64

## 3. Data Preprocessing

In [ ]:
exclude_stops = {'not', 'no', 'nor', "don't", "won't", "didn't", "isn't", "aren't"}
stop_words = set(ENGLISH_STOP_WORDS) - exclude_stops
stemmer = PorterStemmer() # so'zlarni ildiz shakliga keltiradi (running -> run, happier -> happi)

def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    words = text.split()
    clean_words = [stemmer.stem(w) for w in words if w not in stop_words]
    return " ".join(clean_words)

df['clean_basic'] = df['tweet'].apply(clean_text)
df.head()

,tweet,label,clean_basic
0,How unhappy some dogs like it though,negative,unhappi dog like
1,talking to my over driver about where I'm goin...,negative,talk driver im goingh said hed love new york t...
2,Does anybody know if the Rand's likely to fall...,negative,doe anybodi know rand like fall dollar got mon...
3,I miss going to gigs in Liverpool unhappy,negative,miss go gig liverpool unhappi
4,There isnt a new Riverdale tonight ? unhappy,negative,isnt new riverdal tonight unhappi


## 4. Vectorization

In [ ]:
df_unique = df.drop_duplicates(subset='tweet').reset_index(drop=True)
y = df_unique['label']

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))   # matnni so vektorga aylantiradi
# ngram_range - so'zlar ketma-ketligi. (1,2) bu unigrams (bir so'z) va bigrams (ikki so'z) ni olishni bildiradi.

X = vectorizer.fit_transform(df_unique['clean_basic'])

## 4. Train / Test Split (80/20)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=21)

In [26]:
# Naive Bayes, text classification uchun ishlatiladi (spam detection, sentiment analysis).
multinomial = MultinomialNB()
multinomial.fit(X_train, y_train)
y_pred = multinomial.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Naive Bayes Accuracy:", f"{accuracy:.4f}")
print(classification_report(y_test, y_pred))
# precision	model to‘g‘ri topganlar ulushi
# recall	haqiqiy classni topish darajasi
# f1-score	precision va recall o‘rtachasi
# support	shu class nechta

Naive Bayes Accuracy: 0.8596
              precision    recall  f1-score   support

    negative       0.84      0.83      0.83       196
     neutral       0.86      0.95      0.90       295
    positive       0.89      0.76      0.82       200

    accuracy                           0.86       691
   macro avg       0.86      0.85      0.85       691
weighted avg       0.86      0.86      0.86       691



In [27]:
logistic = LogisticRegression(C=2.0, max_iter=1000, n_jobs=-1) # C=2.0 regulyarizatsiyani kuchaytiradi
logistic.fit(X_train, y_train)
y_pred = logistic.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred)
print("Logistic Regression Accuracy:", f"{accuracy_lr:.4f}")
print(classification_report(y_test, y_pred))

Logistic Regression Accuracy: 0.8654
              precision    recall  f1-score   support

    negative       0.94      0.79      0.86       196
     neutral       0.81      0.99      0.89       295
    positive       0.92      0.76      0.83       200

    accuracy                           0.87       691
   macro avg       0.89      0.85      0.86       691
weighted avg       0.88      0.87      0.86       691



In [28]:
svm = LinearSVC() # bu SVMning tezroq versiyasi, text classification uchun yaxshi ishlidi
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
accuracy_svm = accuracy_score(y_test, y_pred)
print("SVM accuracy:", f"{accuracy_svm:.4f}")
print(classification_report(y_test, y_pred))

SVM accuracy: 0.8669
              precision    recall  f1-score   support

    negative       0.91      0.79      0.85       196
     neutral       0.84      0.98      0.90       295
    positive       0.88      0.78      0.82       200

    accuracy                           0.87       691
   macro avg       0.88      0.85      0.86       691
weighted avg       0.87      0.87      0.86       691



## 5. Similarity — Top 10 Most Similar Tweet Pairs


In [29]:
sim_matrix = cosine_similarity(X) # cosine_similarity - bu funksiya matritsaning har bir qatori orasidagi kosinus o'xshashligini hisoblaydi.

pairs = []

for i in range(len(sim_matrix)):
    for j in range(i+1, len(sim_matrix)):
        pairs.append((sim_matrix[i][j], i, j))

pairs = sorted(pairs, reverse=True)

top10 = pairs[:10]

for score,i,j in top10:
    
    print("Similarity:", round(score,3))
    print("Tweet 1:", df['tweet'][i])
    print("Tweet 2:", df['tweet'][j])
    print()

Similarity: 1.0
Tweet 1: have a great day today happy
Tweet 2: ABC News got ahold of me and asked if they could use a few seconds of my coming out video for a transgender special this f

Similarity: 1.0
Tweet 1: appeals people to vote
Tweet 2: ABC News got ahold of me and asked if they could use a few seconds of my coming out video for a transgender special this f

Similarity: 1.0
Tweet 1: appeals people to vote
Tweet 2: have a great day today happy

Similarity: 1.0
Tweet 1: .Thanks so much for the RT
Tweet 2: find out more and sign up at

Similarity: 1.0
Tweet 1: please! Watch the AFL Anzac Game on Tuesday! It's a VERY big tradition here and you'll love the sport! happy
Tweet 2: and have yourself a groovy Thursday!  happy

Similarity: 1.0
Tweet 1: Share the love you're top engaged community members this week! Much Appreciated happy
Tweet 2: and have yourself a groovy Thursday!  happy

Similarity: 1.0
Tweet 1: Share the love you're top engaged community members this week! Much Apprecia

## 7. Word2Vec (from scratch — Skip-gram with Negative Sampling)


In [ ]:
w2v_model_large = Word2Vec(
    [tweet.split() for tweet in df['clean_basic']],
    vector_size=200, window=7, min_count=2, workers=4, sg=1, epochs=20
) # so'zlarni raqamli vektorlarga aylantiradi

# Create vectors with larger Word2Vec
def tweet_vector_large(tweet):
    words = tweet.split()
    vecs = [w2v_model_large.wv[w] for w in words if w in w2v_model_large.wv]
    if len(vecs) == 0:
        return np.zeros(200)
    return np.mean(vecs, axis=0)

X_w2v_large = np.array([tweet_vector_large(tweet) for tweet in df_unique['clean_basic']])

# Normalize and combine with TF-IDF
scaler = StandardScaler() # vektorlarni bir xil scale ga keltiradi
X_w2v_large_scaled = scaler.fit_transform(X_w2v_large)
tfidf_matrix = vectorizer.transform(df_unique['clean_basic'])
X_combined_large = hstack([tfidf_matrix, csr_matrix(X_w2v_large_scaled)]) # Bu ikki feature ni birlashtiradi

# Split combined features
X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(
    X_combined_large, y, test_size=0.2, stratify=y, random_state=42
)

# Hyperparameter tuning
param_grid = {'C': [0.1, 0.5, 1.0, 2.0, 5.0], 'gamma': ['scale', 'auto', 0.1, 0.001]}
svm_grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=3, n_jobs=-1)
svm_grid.fit(X_train_large, y_train_large)

# Base learners
base_learners = [
    ('svm', SVC(kernel='rbf', C=svm_grid.best_params_['C'], gamma=svm_grid.best_params_['gamma'], probability=True)),
    ('lr', LogisticRegression(C=2.0, max_iter=2000, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=150, max_depth=7, learning_rate=0.08, random_state=42, subsample=0.8))
]

# Meta-learner
meta_learner = LogisticRegression(max_iter=2000, C=1.0)

# Create and train stacking model
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5 
) # bir nechta model natijalarini birlashtiradi.

stacking_clf.fit(X_train_large, y_train_large)
y_pred_stacking = stacking_clf.predict(X_test_large)
accuracy_stacking = accuracy_score(y_test_large, y_pred_stacking)

print(f"Stacking Ensemble Accuracy: {accuracy_stacking:.4f}")
print(classification_report(y_test_large, y_pred_stacking))

Stacking Ensemble Accuracy: 0.8726
              precision    recall  f1-score   support

    negative       0.94      0.82      0.87       196
     neutral       0.84      0.94      0.89       295
    positive       0.86      0.83      0.84       200

    accuracy                           0.87       691
   macro avg       0.88      0.86      0.87       691
weighted avg       0.88      0.87      0.87       691

